# Buried: аудит исходных масок
## tl;dr
В подготовленной коллекции 100 semantic-track images. Явный тип Buried есть в 12 masks Mars-Bench. Получено 39 connected-component proposals, не 39 подтверждённых кратеров. Семь проходят начальные size/frame proxy checks. Остальные 88 источников без этого типа не являются отрицательными примерами. Ни новые объекты, ни QA targets не создаются.


## Context & Methods
Source: https://zenodo.org/records/15494713 ; author description: https://arxiv.org/html/2510.24010v1#S0.SS2.SSS2.p5 .
### Key Assumptions
Only source Mirali33/mb-crater_multi_seg and exact local taxonomy Background/Other/Layered/Buried/Secondary are admitted to subtype extraction. Class 3 is Buried. Extract 8-connected regions; mask fragments/merged objects remain proposals. Contact with a different class is not proof of physical superposition. Dimensions, class values and PNG SHA-256 are checked. No models, downloads, training or data_rover access.
The module and its CPU tests provide the computation. Native Jupyter-kernel execution requires ipykernel and nbconvert, which are absent in the current local environment; code cells can be checked sequentially with the existing CPU interpreter. To execute in Jupyter: install nbconvert and ipykernel into .venv-benchmark, register a kernel named planetary-benchmark, then run `python -m jupyter nbconvert --execute --to notebook --ExecutePreprocessor.kernel_name=planetary-benchmark --output buried_mask_audit.executed.ipynb notebooks/buried_mask_audit.ipynb`.


## Data
### 1. Load the saved audit and fixed rules


In [ ]:
from pathlib import Path
import json, hashlib, sys, tomllib
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'configs/crater_annotation_v1.toml').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'configs/crater_annotation_v1.toml').is_file(), 'Set PROJECT_ROOT explicitly to the article repository'
assert 'data_rover' not in {part.casefold() for part in PROJECT_ROOT.parts}
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from planetary_vlm.datasets.crater_audit import components, SOURCE
from planetary_vlm.datasets.orbital_questions import orbital_path
from planetary_vlm.io import read_jsonl
from PIL import Image
import numpy as np
AUDIT_ROOT = PROJECT_ROOT / 'data_orbital/buried_crater_audit_v1'
summary = json.loads((AUDIT_ROOT / 'summary.json').read_text(encoding='utf-8'))
candidates = json.loads((AUDIT_ROOT / 'candidates.json').read_text(encoding='utf-8'))
rule_path = PROJECT_ROOT / 'configs/crater_annotation_v1.toml'
rules = tomllib.loads(rule_path.read_text(encoding='utf-8'))
assert hashlib.sha256(rule_path.read_bytes()).hexdigest() == summary['annotation_rules_sha256']


### 2. Verify source bytes and recompute proposals
This cell is read-only. It never writes masks or rebuilds an existing version.


In [ ]:
recomputed = []
source_count = 0
for relative, expected in summary['source_manifests_sha256'].items():
    manifest_path = orbital_path(PROJECT_ROOT, relative)
    assert hashlib.sha256(manifest_path.read_bytes()).hexdigest() == expected
    for sample in read_jsonl(manifest_path):
        source_count += 1
        for field in ('image', 'mask'):
            asset = orbital_path(PROJECT_ROOT, sample[field])
            assert hashlib.sha256(asset.read_bytes()).hexdigest() == sample[field + '_sha256']
        if sample['source_dataset'] != SOURCE or sample['classes'] != ['Background','Other','Layered','Buried','Secondary']:
            continue
        with Image.open(orbital_path(PROJECT_ROOT, sample['mask'])) as mask_image:
            mask = np.array(mask_image)
        proposals = components(mask, 3, size_fraction=rules['objects']['minimum_short_side_fraction'], margin=rules['visibility']['image_border_margin_px'])
        recomputed.extend({'sample_id': sample['sample_id'], **proposal} for proposal in proposals)
assert source_count == summary['examined_semantic_images']
assert len(recomputed) == len(candidates) == summary['buried_component_candidates']
saved = [(c['source_sample_id'], c['mask_bbox_xywh'], c['mask_pixels'], c['mask_bbox_passes_size_proxy'], c['mask_bbox_within_frame_margin'], c['touches_different_foreground_class']) for c in candidates]
actual = [(c['sample_id'], c['mask_bbox_xywh'], c['mask_pixels'], c['mask_bbox_passes_size_proxy'], c['mask_bbox_within_frame_margin'], c['touches_different_foreground_class']) for c in recomputed]
assert saved == actual
assert all(c['reference_admitted'] is False for c in candidates)
print('Verified source bytes and', len(candidates), 'component proposals, not crater instances')


## Results
### 3. Inspect a bounded list of size/frame candidates


In [ ]:
review_first = [c for c in candidates if c['mask_bbox_passes_size_proxy'] and c['mask_bbox_within_frame_margin']]
assert len(review_first) == summary['size_and_frame_proxy_passing_candidates']
print({key: summary[key] for key in ('examined_semantic_images','subtype_encoded_images','buried_component_candidates','size_and_frame_proxy_passing_candidates','confirmed_instances')})
for candidate in review_first[:10]:
    print(candidate['candidate_id'], candidate['mask_bbox_xywh'])


## Takeaways
Buried is a source crater subtype, not a label for any touching/overlapping objects. Keep source-labelled candidates for review; do not remove them only because they are Buried. Existing full-frame, size and uncertainty gates still apply. The proposal box is not a reconstructed hidden rim. The gallery is `data_orbital/buried_crater_audit_v1/index.html`. No final question labels or independent-instance count are inferred from these masks.
